In [1]:
# Customer Query Intent Classification
# Notebook 1: Data Creation and Cleaning
# Author: Vincent Kometsi

import pandas as pd
import numpy as np
import re
import os

In [2]:
# Create project folders inside the current notebook folder

import os

folders = [
    "data",
    "outputs",
    "outputs/figures",
    "outputs/tables"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully.")

Project folders created successfully.


In [3]:
# Synthetic customer query data

queries = [
    # Claim enquiry
    ("I submitted my claim last week but I have not received any feedback.", "claim_enquiry"),
    ("Can you please check the status of my medical claim?", "claim_enquiry"),
    ("My claim was rejected and I do not understand why.", "claim_enquiry"),
    ("I uploaded my claim documents yesterday. Has it been processed?", "claim_enquiry"),
    ("Please assist me with tracking my hospital claim.", "claim_enquiry"),
    ("I need help submitting a claim for my doctor's visit.", "claim_enquiry"),
    ("Why is my claim still pending after two weeks?", "claim_enquiry"),
    ("I want to know whether my dental claim has been approved.", "claim_enquiry"),
    ("My pharmacy claim did not go through. Please advise.", "claim_enquiry"),
    ("Can someone explain the outcome of my claim assessment?", "claim_enquiry"),

    # Policy update
    ("I want to update my contact details on my policy.", "policy_update"),
    ("Please change my residential address on the system.", "policy_update"),
    ("How do I add a dependent to my medical aid policy?", "policy_update"),
    ("I need to remove a beneficiary from my policy.", "policy_update"),
    ("Can I update my banking details online?", "policy_update"),
    ("Please help me change my surname after marriage.", "policy_update"),
    ("I want to update my email address and phone number.", "policy_update"),
    ("How can I change my plan option for next year?", "policy_update"),
    ("I need to update my employer details.", "policy_update"),
    ("Please assist me with correcting my personal information.", "policy_update"),

    # Benefit question
    ("What benefits do I have for specialist consultations?", "benefit_question"),
    ("Am I covered for physiotherapy sessions?", "benefit_question"),
    ("How many dental visits are covered by my plan?", "benefit_question"),
    ("Does my medical aid cover chronic medication?", "benefit_question"),
    ("I want to know my available hospital benefits.", "benefit_question"),
    ("What maternity benefits are included in my plan?", "benefit_question"),
    ("Does my plan cover mental health consultations?", "benefit_question"),
    ("How much day-to-day benefit do I have left?", "benefit_question"),
    ("Are eye tests covered under my current plan?", "benefit_question"),
    ("Please explain my oncology benefit cover.", "benefit_question"),

    # Complaint
    ("I am unhappy with the service I received from your call centre.", "complaint"),
    ("I have been waiting too long for a response.", "complaint"),
    ("Your consultant gave me incorrect information.", "complaint"),
    ("I want to lodge a formal complaint about my claim.", "complaint"),
    ("The service has been very disappointing.", "complaint"),
    ("Nobody has responded to my emails.", "complaint"),
    ("I was treated unfairly during my previous call.", "complaint"),
    ("I am frustrated because my issue is still unresolved.", "complaint"),
    ("This delay is unacceptable and I need urgent feedback.", "complaint"),
    ("I want to escalate this matter to management.", "complaint"),

    # Payment issue
    ("My debit order did not go through this month.", "payment_issue"),
    ("Why was I charged twice for my monthly premium?", "payment_issue"),
    ("I need help with a failed payment.", "payment_issue"),
    ("My account shows that my premium is overdue.", "payment_issue"),
    ("Please send me my latest payment statement.", "payment_issue"),
    ("I want to query the amount deducted from my bank account.", "payment_issue"),
    ("How do I make a manual payment?", "payment_issue"),
    ("My payment is not reflecting on my profile.", "payment_issue"),
    ("Can you confirm whether my premium has been received?", "payment_issue"),
    ("I need assistance with updating my debit order date.", "payment_issue"),

    # General information request
    ("Where can I find the nearest branch?", "general_information"),
    ("What are your operating hours?", "general_information"),
    ("How do I contact customer support?", "general_information"),
    ("Can you send me information about your medical aid plans?", "general_information"),
    ("Where can I download the mobile app?", "general_information"),
    ("How do I register for the online portal?", "general_information"),
    ("Please provide general information about your services.", "general_information"),
    ("Where can I find the list of network doctors?", "general_information"),
    ("How do I speak to a consultant?", "general_information"),
    ("Can I get information about the available products?", "general_information"),
]

df = pd.DataFrame(queries, columns=["query", "intent"])

df.head()

,query,intent
0,I submitted my claim last week but I have not ...,claim_enquiry
1,Can you please check the status of my medical ...,claim_enquiry
2,My claim was rejected and I do not understand ...,claim_enquiry
3,I uploaded my claim documents yesterday. Has i...,claim_enquiry
4,Please assist me with tracking my hospital claim.,claim_enquiry


In [4]:
# Add simple variations to increase dataset size

prefixes = [
    "",
    "Good day, ",
    "Hi, ",
    "Hello, ",
    "Please assist, ",
    "I need help. "
]

expanded_rows = []

for query, intent in queries:
    for prefix in prefixes:
        expanded_rows.append((prefix + query, intent))

df = pd.DataFrame(expanded_rows, columns=["query", "intent"])

print("Dataset shape:", df.shape)
df.head(10)

Dataset shape: (360, 2)


,query,intent
0,I submitted my claim last week but I have not ...,claim_enquiry
1,"Good day, I submitted my claim last week but I...",claim_enquiry
2,"Hi, I submitted my claim last week but I have ...",claim_enquiry
3,"Hello, I submitted my claim last week but I ha...",claim_enquiry
4,"Please assist, I submitted my claim last week ...",claim_enquiry
5,I need help. I submitted my claim last week bu...,claim_enquiry
6,Can you please check the status of my medical ...,claim_enquiry
7,"Good day, Can you please check the status of m...",claim_enquiry
8,"Hi, Can you please check the status of my medi...",claim_enquiry
9,"Hello, Can you please check the status of my m...",claim_enquiry


In [5]:
df["intent"].value_counts()

intent
claim_enquiry          60
policy_update          60
benefit_question       60
complaint              60
payment_issue          60
general_information    60
Name: count, dtype: int64

In [6]:
# Text cleaning function

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [7]:
df["clean_query"] = df["query"].apply(clean_text)

df.head()

,query,intent,clean_query
0,I submitted my claim last week but I have not ...,claim_enquiry,i submitted my claim last week but i have not ...
1,"Good day, I submitted my claim last week but I...",claim_enquiry,good day i submitted my claim last week but i ...
2,"Hi, I submitted my claim last week but I have ...",claim_enquiry,hi i submitted my claim last week but i have n...
3,"Hello, I submitted my claim last week but I ha...",claim_enquiry,hello i submitted my claim last week but i hav...
4,"Please assist, I submitted my claim last week ...",claim_enquiry,please assist i submitted my claim last week b...


In [18]:
# Save the cleaned dataset

output_path = "data/customer_queries.csv"
df.to_csv(output_path, index=False)

print(f"Dataset saved to: {output_path}")

Dataset saved to: data/customer_queries.csv


In [20]:
df.sample(10, random_state=42)

,query,intent,clean_query
224,"Hi, I am frustrated because my issue is still ...",complaint,hi i am frustrated because my issue is still u...
42,I want to know whether my dental claim has bee...,claim_enquiry,i want to know whether my dental claim has bee...
285,"Hello, My payment is not reflecting on my prof...",payment_issue,hello my payment is not reflecting on my profile
302,"Hi, Where can I find the nearest branch?",general_information,hi where can i find the nearest branch
56,"Hi, Can someone explain the outcome of my clai...",claim_enquiry,hi can someone explain the outcome of my claim...
272,"Hi, I want to query the amount deducted from m...",payment_issue,hi i want to query the amount deducted from my...
15,"Hello, My claim was rejected and I do not unde...",claim_enquiry,hello my claim was rejected and i do not under...
57,"Hello, Can someone explain the outcome of my c...",claim_enquiry,hello can someone explain the outcome of my cl...
248,"Hi, Why was I charged twice for my monthly pre...",payment_issue,hi why was i charged twice for my monthly premium
124,"Please assist, What benefits do I have for spe...",benefit_question,please assist what benefits do i have for spec...
